# Docker Basics: Containers, Images, Volumes


## What is Docker?

Docker is a platform for building, running, and managing **containers**.

A container is a lightweight isolated environment that runs an application with its dependencies.

Example problem without Docker:

- My project needs Python 3.12.
- Another project needs Python 3.9.
- One project needs PostgreSQL 15.
- Another project needs PostgreSQL 13.
- One teammate uses Linux, another uses macOS, another uses Windows.

Docker helps package the app environment so it can run more consistently on different machines.


## Why Docker?

Docker is popular because it helps with:

| Benefit | Meaning |
|---------|---------|
| Portability | The same image can run on many machines. |
| Speed | Containers start faster than full virtual machines. |
| Dependency management | Dependencies live inside the image/container, not directly on the host. |
| Reproducibility | Teammates can run the same environment. |
| Deployment | The same container idea can be used locally and on servers. |
| Isolation | Services can run separately without polluting the host OS. |

Docker does not remove the need to understand Linux, networking, databases, or deployment. It gives us a cleaner way to package and run them.


## Docker vs Virtual Machines

![Virtual machines versus Docker containers](images/docker-vs-vm.svg)

| Topic | Docker Container | Virtual Machine |
|-------|------------------|-----------------|
| OS | Shares host kernel | Runs full guest OS |
| Startup | Usually seconds | Usually slower |
| Size | Usually smaller | Usually larger |
| Isolation | Process-level isolation | Stronger full OS isolation |
| Resource use | Lower overhead | Higher overhead |
| Common use | App packaging and services | Full OS environments |

Simple mental model:

```text
Virtual Machine = full computer inside your computer
Container       = isolated process with its own filesystem and dependencies
```

Docker containers are not exactly the same as VMs. They are lighter because they share the host operating system kernel.


## Docker vs Python Virtual Environment

Python virtual environments solve only Python package isolation:

```bash
python3 -m venv venv
pip install django
```

Docker can isolate more than Python packages:

- Python version
- system libraries
- operating-system packages
- app files
- environment variables
- database service
- Redis service
- Nginx service

| Tool | Solves |
|------|--------|
| `venv` | Python package isolation |
| Docker | Application/service environment isolation |

They can also be used together, but inside a container we often do not need a separate `venv` because the container itself is already isolated.


## Key Concepts

### Image

An **image** is a prepared template used to create containers.

Example images:

```text
ubuntu
python:3.12
postgres:15
redis:7
nginx:alpine
```

Beginner analogy:

```text
Docker image ≈ class / blueprint / template
```

Like a class in programming, an image describes what can be created, but it is not the running thing yet.

### Container

A **container** is a running instance of an image.

```text
image → container
```

Beginner analogy:

```text
Docker container ≈ object / instance created from that image
```

You can create many containers from the same image:

```text
my_django_image → django_container_1
my_django_image → django_container_2
```

Important caveat: this is only an analogy. A Docker image is not really a Python class, and a container is not really a Python object. But it is a useful mental model: **image is the reusable template; container is the running instance**.

### Volume

A **volume** stores data outside the container, so important data can survive when containers are removed.

This is important for databases.

### Docker Hub

**Docker Hub** is a public place for Docker images.

Example:

```bash
docker pull ubuntu
```

Curious note: Docker also has deeper concepts like engine, daemon, registries, networks, and image layers. For now, image/container/volume are enough.


## Installing and Checking Docker

After installing Docker, check the version:

```bash
docker --version
```

Test Docker:

```bash
docker run hello-world
```

If this works, Docker can download an image and run a container.

On Linux, if you get a permission error, your user may not be in the `docker` group. Many servers require either:

```bash
sudo docker ps
```

or adding your user to the Docker group:

```bash
sudo usermod -aG docker $USER
```

Then log out and log in again.


## Working with Images

Search Docker Hub:

```bash
docker search ubuntu
```

Pull an image:

```bash
docker pull ubuntu
```

List local images:

```bash
docker images
```

Remove an image:

```bash
docker rmi ubuntu
```

An image cannot be removed if a container still depends on it. Remove related containers first.


## Running Containers

Run Ubuntu interactively:

```bash
docker run -it ubuntu bash
```

Options:

| Option | Meaning |
|--------|---------|
| `-i` | interactive; keep stdin open |
| `-t` | allocate a terminal |
| `ubuntu` | image name |
| `bash` | command to run inside the container |

Inside the container:

```bash
cat /etc/os-release
pwd
ls
exit
```

When the main process exits, the container stops.

Another friendly test container:

```bash
docker run -d -p 8080:80 docker/welcome-to-docker
```

Then open:

```text
http://localhost:8080
```


## Listing Containers

Running containers:

```bash
docker ps
```

All containers, including stopped ones:

```bash
docker ps -a
```

Useful columns:

| Column | Meaning |
|--------|---------|
| `CONTAINER ID` | Short identifier for the container |
| `IMAGE` | Image used to create it |
| `COMMAND` | Main process/command |
| `STATUS` | Running, exited, etc. |
| `PORTS` | Published ports |
| `NAMES` | Container name |


## Stopping and Removing Containers

Stop a running container:

```bash
docker stop CONTAINER_ID
```

Remove a stopped container:

```bash
docker rm CONTAINER_ID
```

Force remove a running container:

```bash
docker rm -f CONTAINER_ID
```

Remove all stopped containers:

```bash
docker container prune
```

Be careful with prune commands. They delete unused Docker objects.


## Container Logs

View logs:

```bash
docker logs CONTAINER_ID
```

Follow logs live:

```bash
docker logs -f CONTAINER_ID
```

This is similar to checking service logs in deployment. For Dockerized apps, logs are one of the first debugging tools.


## Volumes: Persistent Data

Containers are temporary. If a database container is removed, data inside the container filesystem can be lost.

Use volumes for persistent data:

```bash
docker volume ls
```

Create a volume:

```bash
docker volume create mydata
```

Remove a volume:

```bash
docker volume rm mydata
```

In real projects, PostgreSQL data should use a volume.


## Volumes vs Bind Mounts

For beginners, remember two common ways to keep or share files:

| Type | Simple meaning | Common use |
|------|----------------|------------|
| Volume | Docker-managed storage | Database data, uploaded files |
| Bind mount | Mount a folder from your computer | Live code changes during development |

Example volume for database data:

```bash
docker run -v pgdata:/var/lib/postgresql/data postgres:15
```

Example bind mount for source code:

```bash
docker run -v .:/app my_python_app
```

For this course: use volumes for PostgreSQL data; use bind mounts when you want local code changes to appear inside a container.


## Useful Mental Model

```text
Dockerfile --build--> Image --run--> Container
```

Another beginner analogy:

```text
Class / Blueprint  --create-->  Object / Instance
Docker Image       --run----->  Docker Container
```

For a single small app, one container may be enough.

But real backend projects usually run multiple containers together:

```text
Django container      → web app / API
PostgreSQL container  → database
Redis container       → cache or Celery broker
Celery container      → background tasks
Nginx container       → reverse proxy / static files
```

This is why Docker Compose is important: it lets us run multiple related containers as one project.


## Summary

- Docker runs applications in containers.
- Images are templates; containers are running instances.
- Containers are lighter than virtual machines.
- Docker is broader than Python virtual environments because it packages system-level dependencies too.
- Volumes are used for persistent data.
- Docker Hub is a public registry for images.
- Basic commands: `docker run`, `docker ps`, `docker images`, `docker logs`, `docker stop`, `docker rm`, `docker rmi`.
